# Árboles de decisión y Naive Bayes con Iris

**Autor:** FCO. EMMANUEL DORANTES CAÑA

En este cuaderno se aplican dos clasificadores al conjunto de datos Iris (150 flores, 3 especies, 4 medidas):
1. Árbol de decisión con criterio de **entropía** (la misma idea que ID3 / J48).
2. **Naive Bayes** gaussiano.

## 1. Cargar los datos

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, ConfusionMatrixDisplay

iris = load_iris(as_frame=True)
X = iris.data
y = iris.target
nombres = list(iris.target_names)

df = iris.frame.copy()
df['species'] = df['target'].map(dict(enumerate(nombres)))
df.head()

In [ ]:
print(df.shape)
print(df['species'].value_counts())
df.describe()

## 2. Dividir en entrenamiento (70%) y prueba (30%)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=123, stratify=y)
print("Entrenamiento:", X_train.shape, " Prueba:", X_test.shape)

## 3. Árbol de decisión (entropía)

`criterion='entropy'` hace que el árbol elija cada división con la **ganancia de información**, igual que ID3.

In [ ]:
arbol = DecisionTreeClassifier(criterion='entropy', random_state=123)
arbol.fit(X_train, y_train)

pred_arbol = arbol.predict(X_test)
print("Precisión del árbol:", round(accuracy_score(y_test, pred_arbol), 3))

plt.figure(figsize=(14, 8))
plot_tree(arbol, feature_names=list(X.columns), class_names=nombres, filled=True)
plt.show()

In [ ]:
print(confusion_matrix(y_test, pred_arbol))
print(classification_report(y_test, pred_arbol, target_names=nombres))
ConfusionMatrixDisplay.from_predictions(y_test, pred_arbol, display_labels=nombres)
plt.title("Matriz de confusión - Árbol de decisión")
plt.show()

## 4. Naive Bayes gaussiano

Cada atributo se modela con una distribución normal por clase, y se elige la clase con mayor probabilidad:

$$P(c \mid x) \propto P(c)\prod_i P(x_i \mid c)$$

In [ ]:
nb = GaussianNB()
nb.fit(X_train, y_train)

pred_nb = nb.predict(X_test)
print("Precisión de Naive Bayes:", round(accuracy_score(y_test, pred_nb), 3))
print("Probabilidades a priori:", nb.class_prior_)

In [ ]:
print(confusion_matrix(y_test, pred_nb))
print(classification_report(y_test, pred_nb, target_names=nombres))
ConfusionMatrixDisplay.from_predictions(y_test, pred_nb, display_labels=nombres)
plt.title("Matriz de confusión - Naive Bayes")
plt.show()

## 5. Predecir flores nuevas

In [ ]:
nuevas = pd.DataFrame({
    'sepal length (cm)': [5.1, 6.7, 5.8],
    'sepal width (cm)':  [3.5, 3.0, 2.7],
    'petal length (cm)': [1.4, 5.2, 4.2],
    'petal width (cm)':  [0.2, 2.3, 1.3],
})
print("Árbol:      ", [nombres[i] for i in arbol.predict(nuevas)])
print("Naive Bayes:", [nombres[i] for i in nb.predict(nuevas)])
print(nb.predict_proba(nuevas).round(3))

## 6. Conclusiones

(Escribe aquí con tus palabras qué precisión obtuvo cada modelo, cuál funcionó mejor y en qué especies se equivocaron más. Normalmente los errores están entre *versicolor* y *virginica*, porque sus medidas se traslapan.)